# Evaluating the Updated HYDRA Model

This notebook demonstrates the execution of our `UpdatedHydra` architecture on a single dataset from the UCR archive ("Coffee"). 

The original HYDRA implementation suffered from two main issues:
1. **Apple Silicon (MPS) Incompatibility:** Dynamically generated PyTorch tensors failed to migrate to GPU memory correctly.
2. **Memory Bottlenecks:** Fixed batch sizes caused severe cache-spilling and latency spikes on long sequences.

We resolve these by registering the weights as a PyTorch buffer and implementing a dynamic batching strategy (targeting 4,000 sequence elements).

In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from experiments.utils import load_dataset, get_torch_device

from sklearn.linear_model import RidgeClassifierCV

### 1. Defining the Updated HYDRA Architecture

Below is the `UpdatedHydra` class. Notice the inclusion of `self.register_buffer("W", W)` which ensures device migrations (`.to("mps")` or `.to("cpu")`) safely carry over the weights. We also updated the `.batch()` method to dynamically chunks time series to maximise GPU utilisation.

In [2]:
# Angus Dempster, Daniel F Schmidt, Geoffrey I Webb

# HYDRA: Competing Convolutional Kernels for Fast and Accurate Time Series Classification
# https://arxiv.org/abs/2203.13652

class UpdatedHydra(nn.Module):
    def __init__(self, input_length, k=8, g=64, seed=None):
        super().__init__()

        if seed is not None:
            torch.manual_seed(seed)

        self.k = k # num kernels per group
        self.g = g # num groups

        max_exponent = np.log2((input_length - 1) / (9 - 1)) # kernel length = 9

        self.dilations = 2 ** torch.arange(int(max_exponent) + 1)
        self.num_dilations = len(self.dilations)

        self.paddings = torch.div((9 - 1) * self.dilations, 2, rounding_mode="floor").int()

        self.divisor = min(2, self.g)
        self.h = self.g // self.divisor

        # Create weights
        W = torch.randn(self.num_dilations, self.divisor, self.k * self.h, 1, 9)
        W = W - W.mean(-1, keepdims=True)
        W = W / W.abs().sum(-1, keepdims=True)

        # By registering this as a buffer, PyTorch automatically handles
        # moving it when .to("mps") or .to("cpu") is called later.
        self.register_buffer("W", W)

        # Original code:
        # self.W = torch.randn(self.num_dilations, self.divisor, self.k * self.h, 1, 9)
        # self.W = self.W - self.W.mean(-1, keepdims=True)
        # self.W = self.W / self.W.abs().sum(-1, keepdims=True)
        # But this doesn't move the weights to the correct device when .to("mps") is called, causing errors. 
        # By registering W as a buffer, we ensure it moves correctly.

    # transform in batches of *batch_size*
    def batch(self, X, 
              batch_size: int| None = None, 
              target_elements=4_000):
        """
        Batch the input X to avoid bottlenecks on Apple Silicon MPS when processing large inputs. 
        If batch_size is not provided, we use a custom heuristic to determine an appropriate batch size 
        based on the target number of elements and the sequence length.

        We empirically found target_elements work better than a static batch size, 
        as it allows for larger batches on shorter sequences while preventing cache spilling on longer sequences.

        From our experiments, we found a target of 4,000 works well across the UCR datasets on MPS (M4) and 5,000 on CPU.
        Note that we didn't test other GPU or M chip architectures, so you may want to adjust this target if you're using a different setup.
        """
        if batch_size is None:
            # Our custom Apple Silicon SLC-aware batching
            seq_length = X.shape[-1]

            # We learned that batch sizes that are too big cause cache spilling on MPS
            batch_size = max(1, min(256, target_elements // seq_length))
        num_examples = X.shape[0]
        if num_examples <= batch_size:
            return self(X)
        else:
            Z = []
            for batch in X.split(batch_size, dim=0):
                Z.append(self(batch))
            return torch.cat(Z)


    def forward(self, X):
        # Overridden forward pass using our python lists to prevent .item() stalls
        num_examples = X.shape[0]

        if self.divisor > 1:
            diff_X = torch.diff(X)

        Z = []

        for dilation_index in range(self.num_dilations):
            # Using our lists instead of tensor.item()!
            d = self.dilations[dilation_index].item()
            p = self.paddings[dilation_index].item()

            for diff_index in range(self.divisor):
                _Z = F.conv1d(X if diff_index == 0 else diff_X,
                              self.W[dilation_index, diff_index],
                              dilation=d, padding=p).view(num_examples, self.h, self.k, -1)

                max_values, max_indices = _Z.max(2)
                # Original code:
                # count_max = torch.zeros(num_examples, self.h, self.k)
                count_max = torch.zeros(num_examples, self.h, self.k, device=X.device)
                count_max.scatter_add_(-1, max_indices, max_values)

                min_values, min_indices = _Z.min(2)
                count_min = torch.zeros(num_examples, self.h, self.k, device=X.device)
                # Original code:
                # count_min = torch.zeros(num_examples, self.h, self.k)
                count_min.scatter_add_(-1, min_indices, torch.ones_like(min_values))

                Z.append(count_max)
                Z.append(count_min)

        Z = torch.cat(Z, 1).view(num_examples, -1)
        return Z

### 2. Preprocessing & Utilities
We require a specific scaler to handle the highly sparse feature arrays that HYDRA outputs. We also define quick helper functions to shift data cleanly between `numpy` arrays and `torch` tensors.

In [3]:
class SparseScaler():

    def __init__(self, mask = True, exponent = 4):

        self.mask = mask
        self.exponent = exponent

        self.fitted = False

    def fit(self, X):

        assert not self.fitted, "Already fitted."

        X = X.clamp(0).sqrt()

        self.epsilon = (X == 0).float().mean(0) ** self.exponent + 1e-8

        self.mu = X.mean(0)
        self.sigma = X.std(0) + self.epsilon

        self.fitted = True

    def transform(self, X):

        assert self.fitted, "Not fitted."

        X = X.clamp(0).sqrt()

        if self.mask:
            return ((X - self.mu) * (X != 0)) / self.sigma
        else:
            return (X - self.mu) / self.sigma

    def fit_transform(self, X):

        self.fit(X)

        return self.transform(X)

In [4]:
def _to_tensor(x: np.ndarray, device) -> torch.Tensor:
    """Convert (N, T) numpy array to (N, 1, T) float tensor on the correct device."""
    return torch.from_numpy(x).float().unsqueeze(-2).to(device)

def _to_numpy(t) -> np.ndarray:
    """Safely convert a torch Tensor or any array-like to a numpy array."""
    if isinstance(t, torch.Tensor):
        return t.detach().cpu().numpy()
    return np.asarray(t)

In [5]:
x_train, y_train, x_test, y_test, _ = load_dataset("Coffee")

In [6]:
device = get_torch_device()
print(f"Using device: {device}")

Using device: mps


### 3. Training and Evaluation Pipeline
Finally, we load the dataset, run it through the convolutional dictionaries via our dynamic batching method, scale the features, and fit a Linear Ridge Classifier.

In [7]:
transformer = UpdatedHydra(input_length=x_train.shape[-1], k=8, g=64, seed=42).to(device)
scaler = SparseScaler()

In [8]:
X_train = _to_tensor(x_train, device)
X_test = _to_tensor(x_test, device)

In [9]:
x_train_transformed = transformer.batch(X_train, target_elements=X_train.shape[-1])
x_test_transformed = transformer.batch(X_test, target_elements=X_train.shape[-1])

x_train_scaled = scaler.fit_transform(x_train_transformed)
x_test_scaled = scaler.transform(x_test_transformed)

In [10]:
x_train_np = _to_numpy(x_train_scaled)
x_test_np = _to_numpy(x_test_scaled)
y_train_np = y_train.astype(np.int32)
y_test_np = y_test.astype(np.int32)

In [11]:
classifier = RidgeClassifierCV(alphas=np.logspace(-3, 3, 10))
classifier.fit(x_train_np, y_train_np)

/Users/philiplynch/college/02_semester/ai4ts/assignments/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/philiplynch/college/02_semester/ai4ts/assignments/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/philiplynch/college/02_semester/ai4ts/assignments/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/philiplynch/college/02_semester/ai4ts/assignments/.venv/lib/python3.13/site-packages/sklearn/linear_model/_base.py:312: RuntimeWarning: divide by zero encountered in matmul
  intercept_ = y_offset - X_offset @ coef_
/Users/philiplynch/college/02_semester/ai4ts/assignments/.venv/lib/python3.13/site-packages/sklearn/linear_model/_base.py:312: RuntimeWarning: overflow encountered in matmul
  intercept_ = y_offset - X_offset @ coef_


,alphas,array([1.0000...00000000e+03])
,fit_intercept,True
,scoring,None
,cv,None
,class_weight,None
,store_cv_results,False


In [12]:
classifier.score(x_test_np, y_test_np)

/Users/philiplynch/college/02_semester/ai4ts/assignments/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/philiplynch/college/02_semester/ai4ts/assignments/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/philiplynch/college/02_semester/ai4ts/assignments/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


1.0